# Feature Engineering

This notebook focuses on transforming raw trip data into analytical features that support behavioral analysis of Cyclistic riders.


In [1]:
#importing lib's
import os
from datetime import datetime
import numpy as np 
import pandas as pd

In [2]:
df = pd.read_parquet(r'd:\cicyle project\data_cleaned')

- hours of the day
- weekdays versus weekends
- months and seasons

These features are especially important for identifying behavioral differences between annual members and casual riders, including potential commuting and recreational riding patterns.

### Business Value of Temporal Feature Engineering

This section focuses on extracting granular temporal attributes from the `started_at` timestamp. By transforming a single timestamp into distinct features such as `day_of_week`, `month`, `hour`, `is_weekend` (Boolean), `season` (e.g., 'spring', 'summer'), and `time_period` (e.g., 'morning', 'evening'), we achieve several critical objectives:

1.  **Enhanced Analytical Granularity:** These discrete features allow for highly specific segmentation and analysis of rider behavior. For instance, we can easily compare ride patterns on weekdays versus weekends, or during morning commutes versus evening leisure rides, providing deeper insights into user habits.
2.  **Optimized Query Performance:** In a data warehouse context, splitting date and time components into dedicated dimension tables (e.g., `DIM_DATE`, `DIM_TIME`) significantly improves query performance. Analytical queries involving time-based filtering or aggregation (e.g., "total rides by month," "average ride length by hour") can leverage pre-indexed dimension keys, avoiding complex string parsing or full table scans on large fact tables.
3.  **Star Schema Readiness:** This feature engineering directly supports the design of a Star Schema data warehouse. `started_at` can be used to derive foreign keys to `DIM_DATE` and `DIM_TIME` dimensions, while the original timestamp is retained in the fact table for auditability. This separation of descriptive attributes into dimensions and measurable facts into a central fact table is fundamental to Kimball's dimensional modeling, ensuring both analytical flexibility and efficient data retrieval.
4.  **Reduced Data Redundancy:** Instead of storing redundant date/time components within the main fact table, these attributes are normalized into dimensions. This reduces storage footprint and improves data integrity.

These transformations are crucial for building a robust analytical foundation that can efficiently answer complex business questions about Cyclistic rider behavior and inform strategic decisions.

In [3]:
df["day_of_week"] = df["started_at"].dt.day_name()
df['month'] = df['started_at'].dt.month_name()
df['hour'] = df['started_at'].dt.hour
df['date'] = df['started_at'].dt.date

In [4]:
df['is_weekend'] = df['started_at'].dt.weekday >= 5

In [5]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'fall'

df['season'] = df['started_at'].dt.month.apply(get_season)

In [6]:
def get_time_period(hour):
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour:
        return 'evening'
    else:
        return 'night'
df['time_period'] = df['hour'].apply(get_time_period)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5541823 entries, 0 to 5541822
Data columns (total 21 columns):
 #   Column              Dtype         
---  ------              -----         
 0   ride_id             object        
 1   rideable_type       category      
 2   started_at          datetime64[ns]
 3   ended_at            datetime64[ns]
 4   start_station_name  category      
 5   start_station_id    object        
 6   end_station_name    category      
 7   end_station_id      object        
 8   start_lat           float64       
 9   start_lng           float64       
 10  end_lat             float64       
 11  end_lng             float64       
 12  member_casual       category      
 13  ride_length_min     float64       
 14  day_of_week         object        
 15  month               object        
 16  hour                int32         
 17  date                object        
 18  is_weekend          bool          
 19  season              object        
 20  ti

In [8]:
df.to_parquet('data_feature', index = False)